# Test the agent locally

This notebook runs the **same agent code** that is deployed to AgentCore Runtime (`00-agent-agentcore/agent_files`) on your machine. It builds the Strands `Agent` with `build_agent`, the same function the runtime uses, but **without** AgentCore Memory, so you can iterate on prompts and tools before deploying.

What you need:

* AWS credentials with access to Amazon Bedrock and model access for the model in `MODEL_ID` (default `global.anthropic.claude-sonnet-4-6`).
* Python 3.11 or later.

The cells that only validate payloads run offline. The cells that call the agent need Bedrock.

In [ ]:
%pip install "bedrock-agentcore[strands-agents]==1.25.3" "strands-agents[otel]==1.64.0" twelvelabs==1.3.8 requests==2.34.2 --quiet

## 1. Import the real agent code

Importing `multimodal_agent` reads its settings from environment variables. `BEDROCK_AGENTCORE_MEMORY_ID` is intentionally not set, which turns memory off for local runs. Set `MODEL_ID` before the import to try another model.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.pop("BEDROCK_AGENTCORE_MEMORY_ID", None)
# os.environ["MODEL_ID"] = "global.anthropic.claude-sonnet-4-6"

AGENT_FILES = Path("..") / "00-agent-agentcore" / "agent_files"
sys.path.insert(0, str(AGENT_FILES.resolve()))

import multimodal_agent as ma
from agent_helpers import parse_request

print("Model:", ma.SETTINGS.model_id)
print("Memory enabled:", bool(ma.SETTINGS.memory_id))

## 2. Helper

`ask` follows the same path as the runtime entrypoint: validate the payload with `parse_request`, then build an agent for the request. One agent is created per call, as in production. Local calls therefore do not remember earlier turns, because memory is off. Pass several messages in one call if you want to test multi-turn behavior.

In [ ]:
def ask(prompt, media=None, actor_id="wa-user-5730012345670000000000"):
    payload = {"prompt": prompt, "actor_id": actor_id}
    if media:
        payload["media"] = media
    request = parse_request(payload, "wa-chat-local-test".ljust(33, "0"), None)
    content = ma.build_multimodal_prompt(request.prompt, request.media) if request.media else request.prompt
    agent = ma.build_agent(ma.SETTINGS, request, None)
    result = agent(content)
    print(ma.extract_text(result.message))
    return result

## 3. Offline check: payload validation

These calls never reach a model. They show what the runtime accepts and rejects.

In [ ]:
from agent_helpers import PayloadError

examples = {
    "text": {"prompt": "hola"},
    "empty": {"prompt": "   "},
    "bad image format": {"prompt": "x", "media": {"type": "image", "format": "bmp", "data": "AAAA"}},
    "video without uri": {"prompt": "x", "media": {"type": "video"}},
    "audio transcript": {"prompt": "x", "media": {"type": "audio_transcript", "data": "hola"}},
}
for name, payload in examples.items():
    try:
        request = parse_request(payload, None, None)
        print(f"{name:18} accepted  (channel={request.channel})")
    except PayloadError as error:
        print(f"{name:18} rejected  ({error})")

## 4. Offline check: content block order

The text block must come first, because the memory session manager reads `content[0]["text"]`.

In [ ]:
import base64

blocks = ma.build_multimodal_prompt(
    "Describe this image.",
    {"type": "image", "format": "png", "data": base64.b64encode(b"\x89PNG fake").decode()},
)
print([next(iter(block)) for block in blocks])
assert "text" in blocks[0]

## 5. Text

In [ ]:
_ = ask("Hello! What can you help me with?")

## 6. Image

Use a small image (under 1.5 MB).

In [ ]:
IMAGE_PATH = Path("sample_image.jpg")  # change me

if IMAGE_PATH.exists():
    fmt = {"jpg": "jpeg"}.get(IMAGE_PATH.suffix.lstrip(".").lower(), IMAGE_PATH.suffix.lstrip(".").lower())
    _ = ask(
        "What do you see in this image?",
        media={"type": "image", "format": fmt, "data": base64.b64encode(IMAGE_PATH.read_bytes()).decode()},
    )
else:
    print("Put a small image next to this notebook and update IMAGE_PATH.")

## 7. Audio transcript

In [ ]:
_ = ask(
    "Process this audio.",
    media={"type": "audio_transcript", "data": "Remind me to schedule a meeting next Tuesday at 3 PM with the marketing team."},
)

## 8. Document

In [ ]:
DOCUMENT_PATH = Path("sample_document.pdf")  # change me

if DOCUMENT_PATH.exists():
    _ = ask(
        "Summarize this document.",
        media={
            "type": "document",
            "format": DOCUMENT_PATH.suffix.lstrip("."),
            "data": base64.b64encode(DOCUMENT_PATH.read_bytes()).decode(),
            "name": DOCUMENT_PATH.stem,
        },
    )
else:
    print("Put a small PDF next to this notebook and update DOCUMENT_PATH.")

## 9. Video

The `video_analysis` tool calls the TwelveLabs API directly and needs `TL_SECRET_ARN` (a Secrets Manager secret with a `TL_API_KEY` key) plus an S3 object the tool can presign. Run this cell only if you have both, otherwise use the deployed notebook.

In [ ]:
if os.environ.get("TL_SECRET_ARN") and os.environ.get("S3_BUCKET"):
    _ = ask("What is this video about?", media={"type": "video", "s3_uri": f"s3://{os.environ['S3_BUCKET']}/video/sample_video.mp4"})
else:
    print("TL_SECRET_ARN and S3_BUCKET are not set, skipping the video test.")